In [1]:
!pip install lark pypdf

In [2]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [3]:
import os 
import getpass 
from langchain_aws import BedrockEmbeddings
from langchain_groq import ChatGroq


In [4]:
def set_if_undefined(var:str): 
    if os.environ.get(var): 
        return 
    os.environ[var] = getpass.getpass()
set_if_undefined("GROQ_API_KEY")    

 ········


In [5]:
#Build LLM 
def llm(): 
    model = ChatGroq(
        model = "llama-3.3-70b-versatile", 
        temperature = 0.6, 
        max_tokens = 512, 
        model_kwargs= {
            "top_p": 0.8
        }
    )
    return model

In [6]:
#Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter 
def text_splitter(data, chunk_size, chunk_overlap): 
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, 
        chunk_overlap = chunk_overlap, 
        length_function = len, 
    )
    chunks = splitter.split_documents(data)
    return chunks

In [7]:
#Embedding model 
embedding_model = BedrockEmbeddings(
    model_id = "amazon.titan-embed-text-v2:0", 
    region_name = "us-east-1", 
    model_kwargs = {"dimensions":512}
)
test = embedding_model.embed_query("This is a demo query")
print(len(test))

512


In [8]:
#Retrievers
from langchain_community.document_loaders import TextLoader
loader = TextLoader("companypolicies.txt")
data = loader.load() 
data[0].page_content

"1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violati

In [9]:
chunks_txt = text_splitter(data, 200, 200) 
print(len(chunks_txt))

429


In [10]:
#Storing embeddings in chroma db 
from langchain_community.vectorstores import Chroma
vectordb = Chroma.from_documents(chunks_txt, embedding_model)
print(vectordb._collection.count())

429


In [19]:
#Simple similarity search
query = "Harassment policy" 
retriever = vectordb.as_retriever() 

In [20]:
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='Harassment Policy is a testament to the commitment of this organization in fostering a workplace that is free from discrimination, harassment, and any form of unlawful bias. This policy applies to'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='and Harassment Policy is a testament to the commitment of this organization in fostering a workplace that is free from discrimination, harassment, and any form of unlawful bias. This policy applies'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='8.\tAnti-discrimination and Harassment Policy'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Anti-Discrimination and Harassment Policy is a testament to the commitment of this organization in fostering a workplace that is free from discrimination, harassment, and any form of unlawful bias.')]

In [21]:
retriever = vectordb.as_retriever(search_kwargs = {"k":1})
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='Harassment Policy is a testament to the commitment of this organization in fostering a workplace that is free from discrimination, harassment, and any form of unlawful bias. This policy applies to')]

In [22]:
#MMR retrieval
mmr_retriever = vectordb.as_retriever(search_type = "mmr")
mmr_docs = mmr_retriever.invoke(query) 
mmr_docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='Harassment Policy is a testament to the commitment of this organization in fostering a workplace that is free from discrimination, harassment, and any form of unlawful bias. This policy applies to'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='8.\tAnti-discrimination and Harassment Policy'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Harassment in any form, whether based on the aforementioned characteristics or any other protected status, is unacceptable. This encompasses unwelcome advances, offensive jokes, slurs, and other'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content="each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is")]

In [24]:
#Similarity score threshold retrieval
similarity_score_threshold_retriever = vectordb.as_retriever(search_type = "similarity_score_threshold", search_kwargs ={"score_threshold":0.4})
docs = similarity_score_threshold_retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='Harassment Policy is a testament to the commitment of this organization in fostering a workplace that is free from discrimination, harassment, and any form of unlawful bias. This policy applies to'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='and Harassment Policy is a testament to the commitment of this organization in fostering a workplace that is free from discrimination, harassment, and any form of unlawful bias. This policy applies'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='8.\tAnti-discrimination and Harassment Policy'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Anti-Discrimination and Harassment Policy is a testament to the commitment of this organization in fostering a workplace that is free from discrimination, harassment, and any form of unlawful bias.')]

In [25]:
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf")
pdf_data = loader.load()
pdf_data[1]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you. Its \ncore functionalities encompass: \n1. Context-Aware Capabilities: LangChain facilitates the \ndevelopment of applications that ar

In [26]:
#split 
chunks_pdf = text_splitter(pdf_data, 200,20)
# VectorDB
ids = vectordb.get()["ids"]
vectordb.delete(ids) # We need to delete existing embeddings from previous documents and then store current document embeddings in.
vectordb = Chroma.from_documents(chunks_pdf, embedding_model)

In [28]:
#Multi query Retriever
from langchain_classic.retrievers import MultiQueryRetriever
query = "What does the paper say about langchain?"
retriever = MultiQueryRetriever.from_llm(
    retriever = vectordb.as_retriever(), llm = llm()
)

In [29]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [30]:
docs = retriever.invoke(query)
docs

[Document(metadata={'moddate': '2023-12-31T03:52:06+00:00', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'producer': 'PyPDF', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'page_label': '1', 'total_pages': 6, 'creator': 'Microsoft Word', 'title': 's8329 final', 'page': 0}, page_content='II. LANGCHAIN \nLangChain, with its open -source essence, emerges as a \npromising solution, aiming to simplify the complex process of \ndeveloping applications powered by large language models'),
 Document(metadata={'total_pages': 6, 'page_label': '2', 'author': 'IEEE', 'page': 1, 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'moddate': '2023-12-31T03:52:06+00:00', 'creationdate': '2023-12-31T03:50:13+00:00', 'title': 's8329 final', 'creator': 'Microsoft Word', 'producer': 'PyPDF'}, page_content='2. Reasoning Abilities: LangChain equ

In [32]:
#Self-Querying Retriever 
from langchain_core.documents import Document
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from lark import lark

In [33]:
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]

In [34]:
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]

In [35]:
vectordb = Chroma.from_documents(docs, embedding_model)

In [36]:
document_content_description = "Brief summary of a movie."

retriever = SelfQueryRetriever.from_llm(
    llm(),
    vectordb,
    document_content_description,
    metadata_field_info,
)

In [37]:
# This example only specifies a filter
retriever.invoke("I want to watch a movie rated higher than 8.5")

[Document(metadata={'year': 2006, 'director': 'Satoshi Kon', 'rating': 8.6}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea'),
 Document(metadata={'rating': 9.9, 'genre': 'thriller', 'year': 1979, 'director': 'Andrei Tarkovsky'}, page_content='Three men walk into the Zone, three men walk out of the Zone')]

In [38]:
# This example specifies a query and a filter
retriever.invoke("Has Greta Gerwig directed any movies about women")

[Document(metadata={'director': 'Greta Gerwig', 'rating': 8.3, 'year': 2019}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them')]

In [39]:
# This example specifies a composite filter
retriever.invoke("What's a highly rated (above 8.5) science fiction film?")

[Document(metadata={'rating': 8.6, 'director': 'Satoshi Kon', 'year': 2006}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea'),
 Document(metadata={'rating': 9.9, 'year': 1979, 'genre': 'thriller', 'director': 'Andrei Tarkovsky'}, page_content='Three men walk into the Zone, three men walk out of the Zone')]

In [40]:
# Parent document retriever 
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever 
from langchain_classic.storage import InMemoryStore

In [41]:
# Set two splitters. One is with big chunk size (parent) and one is with small chunk size (child)
parent_splitter = CharacterTextSplitter(chunk_size = 2000, chunk_overlap =20, separator = "\n")
child_splitter = CharacterTextSplitter(chunk_size = 400, chunk_overlap = 20, separator = "\n")

In [42]:
vectordb = Chroma(
    collection_name = "split_parents", embedding_function = embedding_model 
)
store = InMemoryStore()

In [43]:
retriever = ParentDocumentRetriever(
    parent_splitter = parent_splitter, 
    child_splitter = child_splitter, 
    docstore = store,
    vectorstore = vectordb
)

In [44]:
retriever.add_documents(chunks_txt)

In [45]:
len(list(store.yield_keys()))

429

In [46]:
sub_docs = vectordb.similarity_search("smoking policy")

In [47]:
print(sub_docs[0].page_content)

5.	Smoking Policy


In [48]:
retrieved_docs = retriever.invoke("smoking policy")
print(retrieved_docs[0].page_content)

5.	Smoking Policy
